# Curated OCT leaf-refit sensitivity experiment

This notebook trains the curated OCT baseline, refits leaf probabilities on full original imbalanced train split (fixed tree), evaluates validation/test with overridden leaf probabilities, and compares against vanilla and IAI autobalance baselines.

## 1) Setup and split data (same initialization style as competing_methods/train_oct_on_undersampled)

In [1]:
import glob
import os
import numpy as np
import pandas as pd
import importlib

import public.model_IAI
importlib.reload(public.model_IAI)
from public.model_IAI import (
    finetune_oct,
    evaluate_binary_oct,
    get_bin_flag_columns,
    get_true_num_columns,
    train_test_split_enrol,
    refit_leaf_probabilities_on_dataset,
)

TRAIN_TEST_SEED = 123
TARGET_COL = "highcost_gt_200000"
OCT_DEPTHS = [7, 9]
OCT_MINBUCKETS = [50, 100, 120, 150]
OCT_CPS = [0.00001, 0.0001, 0.001, 0.01]

df_og = pd.read_parquet("0917_2017_18_with_2017_cost.parquet")

BIN_FLAG_COLUMNS = get_bin_flag_columns(df_og)
CAT_COLUMNS = df_og.select_dtypes(include=["object", "category"]).columns.tolist()
TRUE_NUM_COLUMNS = get_true_num_columns(df_og, CAT_COLUMNS, BIN_FLAG_COLUMNS)

def make_cost_stratum_3class(df):
    cost_stratum = pd.Series(0, index=df.index)
    cost_stratum[(df["highcost_gt_50000"] == 1) & (df["highcost_gt_100000"] == 0)] = 1
    cost_stratum[(df["highcost_gt_100000"] == 1) & (df["highcost_gt_200000"] == 0)] = 2
    cost_stratum[df["highcost_gt_200000"] == 1] = 3
    return cost_stratum

df_og["cost_stratum_2018"] = make_cost_stratum_3class(df_og)
cutoff_columns = [col for col in df_og.columns if col.startswith("highcost_gt_")]
feature_cols = [
    c for c in df_og.columns
    if c not in (["annual_cost_2017", "annual_cost_2018_deflated", "ENROLID", "cost_stratum_2018"] + cutoff_columns)
]
numeric_cols = df_og[feature_cols + ["cost_stratum_2018"]].select_dtypes(include=["number"]).columns
corrs = df_og[numeric_cols].corr()["cost_stratum_2018"].abs().sort_values(ascending=False)
high_corr_cols = [col for col in corrs[corrs > 0.5].index.tolist() if col != "cost_stratum_2018"]
feature_cols = [col for col in feature_cols if col not in high_corr_cols]

train_ids, test_ids, train_pd, test_pd = train_test_split_enrol(
    df_og,
    target_col="cost_stratum_2018",
    test_size=0.3,
    verbose=False,
    random_state=TRAIN_TEST_SEED,
)
val_ids, test_ids, val_pd, test_pd = train_test_split_enrol(
    test_pd,
    target_col=TARGET_COL,
    test_size=0.5,
    verbose=False,
    random_state=TRAIN_TEST_SEED,
)

X_val = val_pd[feature_cols].copy()
y_val = val_pd[TARGET_COL].copy()
X_test = test_pd[feature_cols].copy()
y_test = test_pd[TARGET_COL].copy()

print(f"Train: {train_pd.shape}, Val: {val_pd.shape}, Test: {test_pd.shape}")
print(f"Feature count: {len(feature_cols)}")

Train: (23479, 96), Val: (5031, 96), Test: (5032, 96)
Feature count: 78


## 2) Train curated OCT on existing curated dataset

In [2]:
CURATED_DATASET_PATH = None
candidate_paths = [
    "/Users/cat2510/my_projects/kcenter_hyperparams_search/kcenter_hyperparameter_search_results_global_seed_123_matching_ratio_1/undersampled_cw_None_pool_True_seed_random.csv"
]
if CURATED_DATASET_PATH is None:
    found = []
    for pattern in candidate_paths:
        found.extend(sorted(glob.glob(pattern)))
    if not found:
        raise FileNotFoundError("Set CURATED_DATASET_PATH to your curated undersampled train CSV.")
    CURATED_DATASET_PATH = found[0]

curated_train_df = pd.read_csv(CURATED_DATASET_PATH)
print("Using curated dataset:", CURATED_DATASET_PATH)
print(curated_train_df[TARGET_COL].value_counts(dropna=False))

curated_model, curated_params, _, curated_preprocessor, curated_feature_names = finetune_oct(
    X_train=curated_train_df[feature_cols],
    y_train=curated_train_df[TARGET_COL],
    X_val=X_val,
    y_val=y_val,
    categorical_cols=CAT_COLUMNS,
    numeric_cols=TRUE_NUM_COLUMNS,
    binary_cols=BIN_FLAG_COLUMNS,
    depths=OCT_DEPTHS,
    minbuckets=OCT_MINBUCKETS,
    cps=OCT_CPS,
)

curated_metrics = evaluate_binary_oct(
    curated_model,
    X_test,
    y_test,
    curated_preprocessor,
    curated_feature_names,
    results_dir="two_stage_kcenter_results_global",
    save_suffix="curated_oct",
    X_val_df=X_val,
    y_val=y_val,
)

print("curated_params:", curated_params)

Using curated dataset: /Users/cat2510/my_projects/kcenter_hyperparams_search/kcenter_hyperparameter_search_results_global_seed_123_matching_ratio_1/undersampled_cw_None_pool_True_seed_random.csv
highcost_gt_200000
1    660
0    660
Name: count, dtype: int64
Finetuning IAI OCT with variants=['oct'], depths=[7, 9], minbuckets=[50, 100, 120, 150], cps=[1e-05, 0.0001, 0.001, 0.01] (best PR-AUC)
→ Building preprocessor w/ conditional imputation:
   • Cat: OHE on: ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
   • Num: scale on: ['stage_2017', 'util_2017', '2017Q1_ckd_cost', '2017Q1_ckd_claims', '2017Q1_max_ckd_stage', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_ckd_claims', '2017Q2_max_ckd_stage', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_ckd_claims', '2017Q3_max_ckd_stage', '2017Q3_d

/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Best params: {'variant': 'oct', 'hyperplane_config': None, 'depth': 7, 'minbucket': 100, 'cp': 0.01, 'best_fit_time_seconds': 2.1734331659972668, 'tuning_time_seconds': 95.57419958314858} @ PR-AUC: 0.1540
Test dataset for OCT application: 5,032 samples


/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


✓ Predictions completed


[ Warning: The type of y (Any) does not match the original target type (Int64)
/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Computing optimal thresholds on validation set (5,031 samples)
  Applied validation-set thresholds to test set for evaluation
✓ Saved OCT predictions to: two_stage_kcenter_results_global/predictions/oct_predictions_curated_oct.csv
✓ Saved split table (5 splits) to: two_stage_kcenter_results_global/oct_tree_curated_oct_splits.csv
✓ Saved OCT tree visualization to: two_stage_kcenter_results_global/oct_tree_curated_oct.html
AUC score: 0.735
PR-AUC (Average Precision): 0.144
Best MCC (test set, threshold from val): 0.275 @ threshold=0.640303
Sensitivity (Recall) @MCC*: 0.289
Specificity @MCC*: 0.981
Balanced (G-mean) recall (test set, threshold from val): 0.669
Balanced (G-mean) specificity (test set, threshold from val): 0.829
Recall @ specificity>=0.60: 0.669 (achieved spec=0.829, thr=0.544715)
Number of leaves: 6
curated_params: {'variant': 'oct', 'hyperplane_config': None, 'depth': 7, 'minbucket': 100, 'cp': 0.01, 'best_fit_time_seconds': 2.1734331659972668, 'tuning_time_seconds': 95.5

### Code to debug if ENROLIDs overlap in train-test-val between undersampled CSV and above split

In [ ]:
#!/usr/bin/env python3
"""
ENROLID overlap check: curated undersampled CSV vs train/val/test from the same
split logic as refit_curated_leaves.ipynb cell 1.

Run from the directory that contains the parquet file (see PARQUET_PATH).
"""
import pandas as pd

import public.model_IAI
from public.model_IAI import get_bin_flag_columns, train_test_split_enrol

TRAIN_TEST_SEED = 123
TARGET_COL = "highcost_gt_200000"
PARQUET_PATH = "0917_2017_18_with_2017_cost.parquet"
CURATED_CSV = (
    "/Users/cat2510/my_projects/kcenter_hyperparams_search/"
    "kcenter_hyperparameter_search_results_global_seed_123_matching_ratio_1/"
    "undersampled_cw_None_pool_False_seed_random.csv"
)


def _norm_enrolid(s):
    if pd.isna(s):
        return None
    return str(int(float(s))) if isinstance(s, (int, float)) else str(s).strip()


def main():
    df_og = pd.read_parquet(PARQUET_PATH)

    BIN_FLAG_COLUMNS = get_bin_flag_columns(df_og)

    def make_cost_stratum_3class(df):
        cost_stratum = pd.Series(0, index=df.index)
        cost_stratum[(df["highcost_gt_50000"] == 1) & (df["highcost_gt_100000"] == 0)] = 1
        cost_stratum[(df["highcost_gt_100000"] == 1) & (df["highcost_gt_200000"] == 0)] = 2
        cost_stratum[df["highcost_gt_200000"] == 1] = 3
        return cost_stratum

    df_og["cost_stratum_2018"] = make_cost_stratum_3class(df_og)
    cutoff_columns = [col for col in df_og.columns if col.startswith("highcost_gt_")]
    feature_cols = [
        c
        for c in df_og.columns
        if c
        not in (
            ["annual_cost_2017", "annual_cost_2018_deflated", "ENROLID", "cost_stratum_2018"]
            + cutoff_columns
        )
    ]
    numeric_cols = df_og[feature_cols + ["cost_stratum_2018"]].select_dtypes(include=["number"]).columns
    corrs = df_og[numeric_cols].corr()["cost_stratum_2018"].abs().sort_values(ascending=False)
    high_corr_cols = [col for col in corrs[corrs > 0.5].index.tolist() if col != "cost_stratum_2018"]
    feature_cols = [col for col in feature_cols if col not in high_corr_cols]

    train_ids, test_ids, train_pd, test_pd = train_test_split_enrol(
        df_og,
        target_col="cost_stratum_2018",
        test_size=0.3,
        verbose=False,
        random_state=TRAIN_TEST_SEED,
    )
    val_ids, test_ids, val_pd, test_pd = train_test_split_enrol(
        test_pd,
        target_col=TARGET_COL,
        test_size=0.5,
        verbose=False,
        random_state=TRAIN_TEST_SEED,
    )

    train_e = {_norm_enrolid(x) for x in train_pd["ENROLID"]}
    val_e = {_norm_enrolid(x) for x in val_pd["ENROLID"]}
    test_e = {_norm_enrolid(x) for x in test_pd["ENROLID"]}
    train_e.discard(None)
    val_e.discard(None)
    test_e.discard(None)

    curated = pd.read_csv(CURATED_CSV)
    if "ENROLID" not in curated.columns:
        raise SystemExit("curated CSV has no ENROLID column")

    cur_e = {_norm_enrolid(x) for x in curated["ENROLID"]}
    cur_e.discard(None)
    dup_cur = int(curated["ENROLID"].duplicated().sum())

    ov_train = cur_e & train_e
    ov_val = cur_e & val_e
    ov_test = cur_e & test_e
    only_outside_train = len(cur_e - train_e)

    print("--- ENROLID overlap diagnostic ---")
    print(f"Parquet: {PARQUET_PATH}")
    print(f"Curated CSV: {CURATED_CSV}")
    print(f"Curated rows: {len(curated)}, unique ENROLIDs: {len(cur_e)}, duplicate rows: {dup_cur}")
    print(f"Overlap curated ∩ train: {len(ov_train)}")
    print(f"Overlap curated ∩ val:   {len(ov_val)}  (expect 0 if curated from train only)")
    print(f"Overlap curated ∩ test:  {len(ov_test)} (expect 0 if curated from train only)")
    print(f"Curated ENROLIDs not in cell-1 train: {only_outside_train}")


if __name__ == "__main__":
    main()


## 3) Refit leaf probabilities from original imbalanced train and run sanity checks

In [6]:
refit_result = refit_leaf_probabilities_on_dataset(
    curated_model,
    train_pd[feature_cols],
    train_pd[TARGET_COL],
    curated_preprocessor,
    curated_feature_names,
)
leaf_prob_map = refit_result["leaf_prob_map"]
fallback_p = refit_result["fallback_probability"]

X_cur_train_proc = curated_preprocessor.transform(curated_train_df[feature_cols])
if hasattr(X_cur_train_proc, "toarray"):
    X_cur_train_proc = X_cur_train_proc.toarray()
X_cur_train_proc = pd.DataFrame(X_cur_train_proc, columns=curated_feature_names)
cur_train_leaves = np.asarray(curated_model.apply(X_cur_train_proc), dtype=int)
cur_train_proba = np.asarray(curated_model.predict_proba(X_cur_train_proc).iloc[:, 1], dtype=float)
orig_leaf_proba_map = pd.DataFrame({"leaf": cur_train_leaves, "p": cur_train_proba}).groupby("leaf")["p"].mean().to_dict()

common_leaves = sorted(set(orig_leaf_proba_map.keys()) & set(leaf_prob_map.keys()))
max_abs_diff = float(max(abs(orig_leaf_proba_map[l] - leaf_prob_map[l]) for l in common_leaves)) if common_leaves else np.nan
mean_abs_diff = float(np.mean([abs(orig_leaf_proba_map[l] - leaf_prob_map[l]) for l in common_leaves])) if common_leaves else np.nan

print("Unique leaves in curated tree (routed by curated train):", len(np.unique(cur_train_leaves)))
print("Leaves with >=1 original-train sample during refit:", len(leaf_prob_map))
print("Global fallback probability:", round(fallback_p, 6))
print("Refit vs original leaf probabilities (common leaves):")
print("  mean abs diff:", round(mean_abs_diff, 6))
print("  max  abs diff:", round(max_abs_diff, 6))


/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Unique leaves in curated tree (routed by curated train): 6
Leaves with >=1 original-train sample during refit: 6
Global fallback probability: 0.02811
Refit vs original leaf probabilities (common leaves):
  mean abs diff: 0.437185
  max  abs diff: 0.566788


In [ ]:
# print per-leaf prevalence (at least rate_curated and rate_full) and mean predict_proba on curated (mean_p_ia_curated) to tie to your existing ~0.40 mean absolute difference.
# For each unique leaf, print:
#   - leaf index
#   - prevalence in curated train set (fraction of samples in this leaf), i.e. rate_curated
#   - prevalence in full train set (fraction of all train samples in this leaf), i.e. rate_full
#   - mean predict_proba on curated set, i.e. mean_p_ia_curated (this is just orig_leaf_proba_map for leaves in curated set)

# First, get the leaves for all train and all curated train
X_full_train_proc = curated_preprocessor.transform(train_pd[feature_cols])
if hasattr(X_full_train_proc, "toarray"):
    X_full_train_proc = X_full_train_proc.toarray()
X_full_train_proc = pd.DataFrame(X_full_train_proc, columns=curated_feature_names)
full_train_leaves = np.asarray(curated_model.apply(X_full_train_proc), dtype=int)

# Count of samples in each leaf (curated and full)
cur_leaf_counts = pd.Series(cur_train_leaves).value_counts().sort_index()
full_leaf_counts = pd.Series(full_train_leaves).value_counts().sort_index()
cur_total = len(cur_train_leaves)
full_total = len(full_train_leaves)

# For leaves present in at least one set, print stats
leaves = sorted(set(cur_leaf_counts.index) | set(full_leaf_counts.index))
print("leaf_id | rate_curated | rate_full | mean_p_ia_curated (predict_proba on curated)")
for leaf in leaves:
    rate_cur = cur_leaf_counts.get(leaf, 0) / cur_total
    rate_full = full_leaf_counts.get(leaf, 0) / full_total
    mean_p_ia_cur = orig_leaf_proba_map.get(leaf, float('nan'))   # might not exist for full_trains not in curated
    print(
        f"{leaf:7d} | "
        f"{rate_cur:12.6f} | "
        f"{rate_full:9.6f} | "
        f"{mean_p_ia_cur:22.6f}"
    )

/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


leaf_ix | rate_curated | rate_full | mean_p_ia_curated (predict_proba on curated)
      6 |     0.312879 |  0.646919 |               0.365617
      7 |     0.093182 |  0.053708 |               0.544715
      8 |     0.107576 |  0.063205 |               0.626761
      9 |     0.153030 |  0.168491 |               0.361386
     10 |     0.116667 |  0.040589 |               0.603896
     11 |     0.216667 |  0.027088 |               0.653846


## 4) Evaluate curated tree + leaf refit on val/test (fixed structure, overridden leaf probabilities)

In [9]:
curated_refit_metrics = evaluate_binary_oct(
    curated_model,
    X_test,
    y_test,
    curated_preprocessor,
    curated_feature_names,
    results_dir="two_stage_kcenter_results_global",
    save_suffix="curated_oct_leaf_refit_original_train",
    X_val_df=X_val,
    y_val=y_val,
    leaf_probability_map=leaf_prob_map,
)

X_val_proc = curated_preprocessor.transform(X_val)
if hasattr(X_val_proc, "toarray"):
    X_val_proc = X_val_proc.toarray()
X_val_proc = pd.DataFrame(X_val_proc, columns=curated_feature_names)
val_leaves = np.asarray(curated_model.apply(X_val_proc), dtype=int)

X_test_proc = curated_preprocessor.transform(X_test)
if hasattr(X_test_proc, "toarray"):
    X_test_proc = X_test_proc.toarray()
X_test_proc = pd.DataFrame(X_test_proc, columns=curated_feature_names)
test_leaves = np.asarray(curated_model.apply(X_test_proc), dtype=int)

missing_val = int(np.sum(~np.isin(val_leaves, list(leaf_prob_map.keys()))))
missing_test = int(np.sum(~np.isin(test_leaves, list(leaf_prob_map.keys()))))

print("Validation samples in leaves missing from refit map:", missing_val)
print("Test samples in leaves missing from refit map:", missing_test)

Test dataset for OCT application: 5,032 samples


/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


✓ Predictions completed
Computing optimal thresholds on validation set (5,031 samples)
  Applied validation-set thresholds to test set for evaluation


/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


✓ Saved OCT predictions to: two_stage_kcenter_results_global/predictions/oct_predictions_curated_oct_leaf_refit_original_train.csv
✓ Saved split table (5 splits) to: two_stage_kcenter_results_global/oct_tree_curated_oct_leaf_refit_original_train_splits.csv
✓ Saved OCT tree visualization to: two_stage_kcenter_results_global/oct_tree_curated_oct_leaf_refit_original_train.html
AUC score: 0.802
PR-AUC (Average Precision): 0.149
Best MCC (test set, threshold from val): 0.275 @ threshold=0.195806
Sensitivity (Recall) @MCC*: 0.289
Specificity @MCC*: 0.981
Balanced (G-mean) recall (test set, threshold from val): 0.669
Balanced (G-mean) specificity (test set, threshold from val): 0.829
Recall @ specificity>=0.60: 0.817 (achieved spec=0.664, thr=0.018453)
Number of leaves: 6
Leaf override active: 6 mapped leaves
Missing mapped leaves in test routing (fallback used): 0
Missing mapped leaves in val routing (fallback used): 0


/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1, 2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Validation samples in leaves missing from refit map: 0
Test samples in leaves missing from refit map: 0


## 5) Baselines (vanilla_oct and iai_autobalance)

If `vanilla_metrics` / `auto_metrics` already exist from `competing_methods.ipynb`, this cell reuses them; otherwise it trains quickly here.

In [5]:
# Symmetric leaf-level excess AUC: vanilla OCT (M_v) vs IAI autobalance OCT (M_s)

import importlib
import os

import symmetric_excess_AUC
importlib.reload(symmetric_excess_AUC)
from symmetric_excess_AUC import symmetric_leaf_evaluation_oct

print("=" * 80)
print("SYMMETRIC LEAF EVALUATION: VANILLA OCT vs CURATED OCT")
print("=" * 80)

symmetric_results = {}


def format_val(val, fmt=".6f"):
    if val is None or (isinstance(val, str) and val == "N/A"):
        return "N/A"
    try:
        if isinstance(val, (int, float)):
            return f"{val:{fmt}}"
        return str(val)
    except (ValueError, TypeError):
        return str(val)

if "TRAIN_TEST_SEED" in globals():
    vanilla_pred_path = f"vanilla_oct/seed_{TRAIN_TEST_SEED}/predictions/oct_predictions.csv"

auto_pred_path = "two_stage_kcenter_results_global/predictions/oct_predictions_curated_oct.csv"
print(f"Vanilla predictions:     {vanilla_pred_path}  (exists={os.path.exists(vanilla_pred_path)})")
print(f"Autobalance predictions:   {auto_pred_path}  (exists={os.path.exists(auto_pred_path)})")

if not os.path.exists(vanilla_pred_path) or not os.path.exists(auto_pred_path):
    raise FileNotFoundError(
        "Need both vanilla and IAI autobalance OCT prediction CSVs. "
        "Run the vanilla OCT and IAI autobalance training cells first."
    )

subgroup_results = symmetric_leaf_evaluation_oct(
    mv_pred_path=vanilla_pred_path,
    ms_pred_path=auto_pred_path,
    y_test=y_test,
)
symmetric_results["vanilla_vs_iai_autobalance"] = subgroup_results

print("\n  Symmetric excess AUC (M_v = vanilla, M_s = IAI autobalance):")
if "scores" in subgroup_results:
    scores = subgroup_results["scores"]
    print(scores)
    print(f"    Excess ROC (autobalance|vanilla): {format_val(scores.get('excess_ROC_s|v', 'N/A'))}")
    print(f"    Excess ROC (vanilla|autobalance): {format_val(scores.get('excess_ROC_v|s', 'N/A'))}")
    print(f"    Symmetric Excess ROC: {format_val(scores.get('sym_excess_ROC', 'N/A'))}")
    print(f"    Coverage (informative vanilla leaves): {format_val(scores.get('coverage_informative_v', 'N/A'), '.4f')}")
    print(f"    Coverage (informative autobalance leaves): {format_val(scores.get('coverage_informative_s', 'N/A'), '.4f')}")


if "overall_ci" in subgroup_results:
    overall_ci = subgroup_results["overall_ci"]
    print("\n  Global metrics (M_v = vanilla, M_s = autobalance):")
    print(f"    Vanilla ROC-AUC:     {format_val(overall_ci.get('global_ROC_Mv', 'N/A'), '.6f')}")
    print(f"    Autobalance ROC-AUC: {format_val(overall_ci.get('global_ROC_Ms', 'N/A'), '.6f')}")
    print(f"    Vanilla PR-AUC:      {format_val(overall_ci.get('global_PR_Mv', 'N/A'), '.6f')}")
    print(f"    Autobalance PR-AUC:  {format_val(overall_ci.get('global_PR_Ms', 'N/A'), '.6f')}")

if "ci" in subgroup_results:
    print("\n  Bootstrap 95% CIs:")
    ci = subgroup_results["ci"]
    if "excess_ROC_s|v" in ci:
        x = ci["excess_ROC_s|v"]
        print(f"    Excess ROC (autobalance|vanilla): [{format_val(x.get('lo', 'N/A'))}, {format_val(x.get('hi', 'N/A'))}]")
    if "excess_ROC_v|s" in ci:
        x = ci["excess_ROC_v|s"]
        print(f"    Excess ROC (vanilla|autobalance): [{format_val(x.get('lo', 'N/A'))}, {format_val(x.get('hi', 'N/A'))}]")



SYMMETRIC LEAF EVALUATION: VANILLA OCT vs CURATED OCT
Vanilla predictions:     vanilla_oct/seed_123/predictions/oct_predictions.csv  (exists=True)
Autobalance predictions:   two_stage_kcenter_results_global/predictions/oct_predictions_curated_oct.csv  (exists=True)

  Symmetric excess AUC (M_v = vanilla, M_s = IAI autobalance):
{'excess_ROC_s|v': 0.17809057738988482, 'excess_ROC_v|s': 0.0037470075980987404, 'sym_excess_ROC': 0.09091879249399178, 'excess_ROC_gap_s|v_minus_v|s': 0.17434356979178608, 'coverage_informative_v': 1.0, 'coverage_informative_s': 1.0}
    Excess ROC (autobalance|vanilla): 0.178091
    Excess ROC (vanilla|autobalance): 0.003747
    Symmetric Excess ROC: 0.090919
    Coverage (informative vanilla leaves): 1.0000
    Coverage (informative autobalance leaves): 1.0000

  Global metrics (M_v = vanilla, M_s = autobalance):
    Vanilla ROC-AUC:     {'point': 0.5912373970855807, 'lo': 0.5601098584620336, 'hi': 0.6236686401013485, 'se_boot': 0.016654048549592265, 'n_eff':

In [ ]:
if "vanilla_metrics" not in globals():
    vanilla_model, vanilla_params, _, vanilla_preprocessor, vanilla_feature_names = finetune_oct(
        X_train=train_pd[feature_cols],
        y_train=train_pd[TARGET_COL],
        X_val=X_val,
        y_val=y_val,
        categorical_cols=CAT_COLUMNS,
        numeric_cols=TRUE_NUM_COLUMNS,
        binary_cols=BIN_FLAG_COLUMNS,
        depths=OCT_DEPTHS,
        minbuckets=OCT_MINBUCKETS,
        cps=OCT_CPS,
    )
    vanilla_metrics = evaluate_binary_oct(
        vanilla_model,
        X_test,
        y_test,
        vanilla_preprocessor,
        vanilla_feature_names,
        results_dir="vanilla_oct/seed_123",
        save_suffix="vanilla_oct",
        X_val_df=X_val,
        y_val=y_val,
    )

if "auto_metrics" not in globals():
    auto_model, auto_params, _, auto_preprocessor, auto_feature_names = finetune_oct(
        X_train=train_pd[feature_cols],
        y_train=train_pd[TARGET_COL],
        X_val=X_val,
        y_val=y_val,
        categorical_cols=CAT_COLUMNS,
        numeric_cols=TRUE_NUM_COLUMNS,
        binary_cols=BIN_FLAG_COLUMNS,
        depths=[7],
        minbuckets=[100],
        cps=[0.01],
        fit_sample_weight="autobalance",
    )
    auto_metrics = evaluate_binary_oct(
        auto_model,
        X_test,
        y_test,
        auto_preprocessor,
        auto_feature_names,
        results_dir="iai_autobalance_oct/seed_123_cp10e-2_depth7_minbucket100",
        save_suffix="iai_autobalance",
        X_val_df=X_val,
        y_val=y_val,
    )


Finetuning IAI OCT with variants=['oct'], depths=[7, 9], minbuckets=[50, 100, 120, 150], cps=[1e-05, 0.0001, 0.001, 0.01] (best PR-AUC)
→ Building preprocessor w/ conditional imputation:
   • Cat: OHE on: ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
   • Num: scale on: ['stage_2017', 'util_2017', '2017Q1_ckd_cost', '2017Q1_ckd_claims', '2017Q1_max_ckd_stage', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_ckd_claims', '2017Q2_max_ckd_stage', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_ckd_claims', '2017Q3_max_ckd_stage', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4_ckd_claims', '2017Q4_max_ckd_stage', '2017Q4_direct_ckd_cost', '2017Q4_procedure_ckd_cost', '2017Q4_comorbidity_ckd_cost', 'ckd_cost_trend_2017', 'ckd_cos

## 6) Final comparison table + interpretation helper

In [ ]:
def _row(method_name, m):
    return {
        "method": method_name,
        "auc": m.get("auc", np.nan),
        "pr_auc": m.get("pr_auc", np.nan),
        "best_mcc": m.get("best_mcc", np.nan),
        "balanced_recall_gmean": m.get("balanced_recall_gmean", np.nan),
        "balanced_specificity_gmean": m.get("balanced_specificity_gmean", np.nan),
        "number_of_leaves": m.get("number_of_leaves", np.nan),
        "number_of_splits": m.get("number_of_splits", np.nan),
    }

comparison_df = pd.DataFrame([
    _row("vanilla_oct", vanilla_metrics),
    _row("curated_oct", curated_metrics),
    _row("curated_oct_leaf_refit_original_train", curated_refit_metrics),
    _row("iai_autobalance", auto_metrics),
])

comparison_df = comparison_df[[
    "method",
    "auc",
    "pr_auc",
    "best_mcc",
    "balanced_recall_gmean",
    "balanced_specificity_gmean",
    "number_of_leaves",
    "number_of_splits",
]]

print(comparison_df.to_string(index=False))

print("\nInterpretation guide:")
print("- If leaf-refit improves curated metrics: probability estimation on curated 1:1 leaves was a bottleneck.")
print("- If leaf-refit barely changes metrics: curated weakness is mostly partition/ranking quality, not leaf-rate calibration.")
print("- If leaf-refit worsens metrics: full-train empirical leaf rates may be less aligned with validation-tuned operating points.")